In [1]:
!pip install -q datasets scikit-learn pandas pyarrow matplotlib

In [2]:
import ast
import json
import re
from collections import Counter

import pandas as pd
from datasets import load_dataset

VAL_URL = (
    "https://huggingface.co/datasets/"
    "EleutherAI/pile_val_test/resolve/main/val.jsonl"
)

# Load as a streaming dataset
stream = load_dataset(
    "json",
    data_files=VAL_URL,
    split="train",
    streaming=True
)

# Approximate randomization without loading everything
stream = stream.shuffle(seed=42, buffer_size=10_000)

In [3]:
def extract_source(meta):
    """Extract the Pile component name from metadata."""

    if isinstance(meta, dict):
        return meta.get("pile_set_name", "Unknown")

    if isinstance(meta, str):
        try:
            parsed = json.loads(meta)
        except (json.JSONDecodeError, TypeError):
            try:
                parsed = ast.literal_eval(meta)
            except (ValueError, SyntaxError):
                return "Unknown"

        if isinstance(parsed, dict):
            return parsed.get("pile_set_name", "Unknown")

    return "Unknown"


def clean_text(text, max_characters=3000):
    """Perform minimal cleaning and truncate long documents."""

    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_characters]

In [4]:
DOCUMENTS_PER_SOURCE = 400
MIN_CHARACTERS = 200
MAX_CHARACTERS = 3000
MAX_RECORDS_TO_SCAN = 200_000

rows = []
source_counts = Counter()

for record_number, example in enumerate(stream):

    if record_number >= MAX_RECORDS_TO_SCAN:
        break

    text = clean_text(
        example.get("text", ""),
        max_characters=MAX_CHARACTERS
    )

    if len(text) < MIN_CHARACTERS:
        continue

    source = extract_source(example.get("meta", {}))

    # Prevent large sources from dominating the sample
    if source_counts[source] >= DOCUMENTS_PER_SOURCE:
        continue

    rows.append({
        "text": text,
        "source": source
    })

    source_counts[source] += 1

df = pd.DataFrame(rows)

print("Number of documents:", len(df))
print("Number of sources:", df["source"].nunique())

display(
    df["source"]
    .value_counts()
    .rename_axis("source")
    .reset_index(name="documents")
)

Number of documents: 6827
Number of sources: 22


,source,documents
0,Pile-CC,400
1,Wikipedia (en),400
2,StackExchange,400
3,PubMed Abstracts,400
4,OpenWebText2,400
5,PubMed Central,400
6,ArXiv,400
7,USPTO Backgrounds,400
8,DM Mathematics,400
9,Github,400


In [5]:
df.to_parquet("pile_sample.parquet", index=False)
df.to_csv("pile_sample.csv", index=False)

In [6]:
df = pd.read_parquet("pile_sample.parquet")

In [7]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

keyword_vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    max_features=20_000,
    sublinear_tf=True
)

keyword_matrix = keyword_vectorizer.fit_transform(df["text"])
terms = np.array(keyword_vectorizer.get_feature_names_out())

print("TF-IDF matrix shape:", keyword_matrix.shape)

TF-IDF matrix shape: (6827, 20000)


In [8]:
mean_scores = np.asarray(keyword_matrix.mean(axis=0)).ravel()
top_indices = mean_scores.argsort()[::-1][:30]

overall_keywords = pd.DataFrame({
    "keyword": terms[top_indices],
    "tfidf_score": mean_scores[top_indices]
})

display(overall_keywords)

,keyword,tfidf_score
0,like,0.011378
1,new,0.011101
2,time,0.010605
3,let,0.010354
4,10,0.010153
5,just,0.009967
6,com,0.009814
7,use,0.009323
8,know,0.009012
9,don,0.008794


In [9]:
def source_keywords(
    dataframe,
    matrix,
    vocabulary,
    top_n=15
):
    results = []

    for source in sorted(dataframe["source"].unique()):
        positions = np.where(
            dataframe["source"].to_numpy() == source
        )[0]

        source_scores = np.asarray(
            matrix[positions].mean(axis=0)
        ).ravel()

        top_positions = source_scores.argsort()[::-1][:top_n]

        for rank, position in enumerate(top_positions, start=1):
            results.append({
                "source": source,
                "rank": rank,
                "keyword": vocabulary[position],
                "score": source_scores[position]
            })

    return pd.DataFrame(results)


source_keyword_df = source_keywords(
    dataframe=df,
    matrix=keyword_matrix,
    vocabulary=terms,
    top_n=15
)

display(source_keyword_df.head(45))

,source,rank,keyword,score
0,ArXiv,1,abstract,0.037110
1,ArXiv,2,author,0.036096
2,ArXiv,3,title,0.035435
3,ArXiv,4,introduction,0.030646
4,ArXiv,5,quantum,0.024458
5,ArXiv,6,model,0.022172
6,ArXiv,7,physics,0.022138
7,ArXiv,8,bib,0.021227
8,ArXiv,9,theory,0.019767
9,ArXiv,10,energy,0.019280


In [10]:
source_keyword_df[
    source_keyword_df["source"] == "ArXiv"
]

,source,rank,keyword,score
0,ArXiv,1,abstract,0.037110
1,ArXiv,2,author,0.036096
2,ArXiv,3,title,0.035435
3,ArXiv,4,introduction,0.030646
4,ArXiv,5,quantum,0.024458
5,ArXiv,6,model,0.022172
6,ArXiv,7,physics,0.022138
7,ArXiv,8,bib,0.021227
8,ArXiv,9,theory,0.019767
9,ArXiv,10,energy,0.019280


In [11]:
source_keyword_df.to_csv(
    "pile_keywords_by_source.csv",
    index=False
)

In [12]:
from sklearn.decomposition import NMF

NUMBER_OF_TOPICS = 15

nmf_model = NMF(
    n_components=NUMBER_OF_TOPICS,
    init="nndsvda",
    random_state=42,
    max_iter=400
)

document_topic_matrix = nmf_model.fit_transform(keyword_matrix)

print("Document-topic shape:", document_topic_matrix.shape)

Document-topic shape: (6827, 15)


In [13]:
def display_nmf_topics(model, feature_names, words_per_topic=12):
    topic_records = []

    for topic_number, component in enumerate(model.components_):
        top_positions = component.argsort()[::-1][:words_per_topic]
        topic_words = feature_names[top_positions]

        topic_records.append({
            "topic": topic_number,
            "keywords": ", ".join(topic_words)
        })

    return pd.DataFrame(topic_records)


topic_summary = display_nmf_topics(
    model=nmf_model,
    feature_names=terms,
    words_per_topic=12
)

display(topic_summary)

,topic,keywords
0,0,"don, know, just, like, ll, right, think, ve, g..."
1,1,"let, suppose, let let, derivative, let derivat..."
2,2,"patients, study, treatment, clinical, disease,..."
3,3,"court, district, appellant, appeals, district ..."
4,4,"return, public, license, class, string, file, ..."
5,5,"enron, ect, hou, hou ect, cc, subject, ect ect..."
6,6,"book, chapter, copyright, books, isbn, york, n..."
7,7,"abstract, title, author, introduction, quantum..."
8,8,"que, la, en, se, el, para, por, es, una, como,..."
9,9,"11, 13, 12, 10, 14, 16, 15, 17, 22, 18, 19, 21"


In [14]:
df["topic"] = document_topic_matrix.argmax(axis=1)
df["topic_strength"] = document_topic_matrix.max(axis=1)

display(
    df[
        ["source", "topic", "topic_strength", "text"]
    ].head()
)

,source,topic,topic_strength,text
0,Pile-CC,4,0.051904,Data-BitStream-0.07 NAME VERSION version 0.03 ...
1,Wikipedia (en),10,0.072810,Thrombolite Thrombolites are ancient forms of ...
2,StackExchange,4,0.080004,Q: Use route as url in config with Symfony I u...
3,Pile-CC,2,0.028575,A mammoth and humans on the banks of the Marne...
4,PubMed Abstracts,14,0.033008,Synthetic polycation: polynucleotide interacti...


In [15]:
topic_counts = (
    df["topic"]
    .value_counts()
    .sort_index()
    .rename_axis("topic")
    .reset_index(name="documents")
)

display(topic_counts)

,topic,documents
0,0,868
1,1,204
2,2,827
3,3,462
4,4,590
5,5,320
6,6,501
7,7,593
8,8,236
9,9,254


In [16]:
source_topic_table = pd.crosstab(
    df["source"],
    df["topic"],
    normalize="index"
).round(3)

display(source_topic_table)

topic,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
source,,,,,,,,,,,,,,,
ArXiv,0.000,0.000,0.002,0.000,0.000,0.000,0.000,0.985,0.005,0.000,0.000,0.000,0.000,0.005,0.002
BookCorpus2,0.042,0.000,0.000,0.000,0.000,0.000,0.917,0.000,0.000,0.042,0.000,0.000,0.000,0.000,0.000
Books3,0.029,0.000,0.011,0.000,0.000,0.000,0.907,0.004,0.014,0.036,0.000,0.000,0.000,0.000,0.000
DM Mathematics,0.000,0.507,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.438,0.000,0.000,0.055,0.000,0.000
Enron Emails,0.045,0.000,0.018,0.008,0.022,0.778,0.012,0.005,0.002,0.015,0.000,0.065,0.000,0.030,0.000
EuroParl,0.008,0.000,0.041,0.017,0.000,0.000,0.017,0.000,0.901,0.008,0.000,0.000,0.000,0.008,0.000
FreeLaw,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
Github,0.010,0.002,0.002,0.002,0.782,0.010,0.005,0.018,0.002,0.045,0.000,0.110,0.000,0.010,0.000
Gutenberg (PG-19),0.070,0.000,0.000,0.000,0.000,0.000,0.895,0.000,0.000,0.035,0.000,0.000,0.000,0.000,0.000


In [17]:
!pip install -q bertopic sentence-transformers umap-learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.6 MB/s eta 0:00:00


In [18]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Use a smaller sample
bertopic_df = df.sample(
    n=min(7000, len(df)),
    random_state=42
).copy()

# Further truncate texts to reduce embedding time
documents = bertopic_df["text"].str[:1500].tolist()

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    min_topic_size=30,
    calculate_probabilities=False,
    low_memory=True,
    verbose=True
)

topics, probabilities = topic_model.fit_transform(documents)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-07-21 07:01:41,698 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/214 [00:00<?, ?it/s]

2026-07-21 07:02:01,923 - BERTopic - Embedding - Completed ✓
2026-07-21 07:02:01,924 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-21 07:02:34,358 - BERTopic - Dimensionality - Completed ✓
2026-07-21 07:02:34,360 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-21 07:02:34,595 - BERTopic - Cluster - Completed ✓
2026-07-21 07:02:34,601 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-21 07:02:36,256 - BERTopic - Representation - Completed ✓


In [19]:
topic_info = topic_model.get_topic_info()
display(topic_info.head(20))

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1366,-1_the_to_is_and,"[the, to, is, and, of, in, what, it, that, for]","[Ask HN: Review my coming soon page, SettleIt...."
1,0,1293,0_of_and_the_in,"[of, and, the, in, to, with, is, for, that, by]",[The objectives of this Project are two-fold. ...
2,1,702,1_the_this_to_import,"[the, this, to, import, if, class, license, pu...",[Q: matlab - what is the equivalent of null / ...
3,2,456,2_court_of_the_district,"[court, of, the, district, for, and, appellant...",[991 F.2d 791 NOTICE: Fourth Circuit I.O.P. 36...
4,3,439,3_the_of_in_we,"[the, of, in, we, and, is, to, for, that, abst...",[--- abstract: 'The dynamical stability of non...
5,4,414,4_you_it_me_to,"[you, it, me, to, my, that, we, what, re, don]","[""Help!"" ""Help!"" "" Somebody help me!"" "" Oh my ..."
6,5,398,5_the_chapter_of_and,"[the, chapter, of, and, in, by, book, to, this...",[Begin Reading Table of Contents About the Aut...
7,6,380,6_enron_ect_hou_to,"[enron, ect, hou, to, com, subject, the, 2001,...",[calendar/mtg folder ----- Forwarded by Steven...
8,7,364,7_the_of_to_and,"[the, of, to, and, in, is, or, invention, an, ...",[1. Field of the Invention The present inventi...
9,8,205,8_to_it_the_and,"[to, it, the, and, that, you, for, is, of, com]",[12 things I learned from pitching VCs this pa...


In [20]:
topic_model.get_topic(0)

[('of', np.float64(0.04161192565061939)),
 ('and', np.float64(0.03923545837345143)),
 ('the', np.float64(0.035775828558398455)),
 ('in', np.float64(0.033587942874474065)),
 ('to', np.float64(0.02625117119510819)),
 ('with', np.float64(0.020257249043064283)),
 ('is', np.float64(0.017130677594756703)),
 ('for', np.float64(0.01650505997187195)),
 ('that', np.float64(0.014942528245137827)),
 ('by', np.float64(0.014484591812792953))]

In [21]:
bertopic_df["topic"] = topics
bertopic_df.to_csv(
    "pile_bertopic_results.csv",
    index=False
)